# 01. 데이터 확인(Data Inspection)

**프로젝트 제목:** 아동학대 의심 예측 설문조사  
**분석 중심:** ASD 선별 관련 행동·개인·배경 특성 분석

이 노트북에서는 데이터 구조, 중복, 결측값, Target, 그리고 Data Leakage를 먼저 확인합니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('Autism-Child-Data.csv')
df_raw = pd.read_csv(DATA_PATH)
print('원본 크기:', df_raw.shape)
display(df_raw.head())
print('컬럼:')
print(df_raw.columns.tolist())

원본 크기: (292, 21)


,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,age,gender,ethnicity,jundice,austim,contry_of_res,used_app_before,result,age_desc,relation,Class/ASD
0,1,1,0,0,1,1,0,1,0,0,6,m,Others,no,no,Jordan,no,5,'4-11 years',Parent,NO
1,1,1,0,0,1,1,0,1,0,0,6,m,'Middle Eastern ',no,no,Jordan,no,5,'4-11 years',Parent,NO
2,1,1,0,0,0,1,1,1,0,0,6,m,?,no,no,Jordan,yes,5,'4-11 years',?,NO
3,0,1,0,0,1,1,0,0,0,1,5,f,?,yes,no,Jordan,no,4,'4-11 years',?,NO
4,1,1,1,1,1,1,1,1,1,1,5,m,Others,yes,no,'United States',no,10,'4-11 years',Parent,YES


컬럼:
['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'age', 'gender', 'ethnicity', 'jundice', 'austim', 'contry_of_res', 'used_app_before', 'result', 'age_desc', 'relation', 'Class/ASD']


## 1) 데이터 품질 확인
- 중복(Duplicate)
- 결측 표시 `?`
- 모든 행에서 값이 같은 상수 변수(Constant Feature)

In [2]:
print('완전 중복 행:', df_raw.duplicated().sum())
question_missing = (df_raw.astype(str) == '?').sum().sort_values(ascending=False)
display(question_missing[question_missing > 0].rename('missing_?').to_frame())

print('age_desc 고유값:', df_raw['age_desc'].unique())
print('age_desc 고유값 개수:', df_raw['age_desc'].nunique(dropna=False))

완전 중복 행: 2


,missing_?
ethnicity,43
relation,43
age,4


age_desc 고유값: ["'4-11 years'"]
age_desc 고유값 개수: 1


## 2) Target 확인
`Class/ASD`가 이번 데이터의 직접 Target이며 YES/NO 분류 문제입니다.

In [3]:
print(df_raw['Class/ASD'].value_counts())
print(df_raw['Class/ASD'].value_counts(normalize=True).round(3))

Class/ASD
NO     151
YES    141
Name: count, dtype: int64
Class/ASD
NO     0.517
YES    0.483
Name: proportion, dtype: float64


## 3) Data Leakage 확인
`result`가 A1~A10 합계인지, 그리고 `result >= 7`이 `Class/ASD=YES`와 일치하는지 확인합니다.

In [4]:
behavior = [f'A{i}_Score' for i in range(1,11)]
sum_match = (df_raw[behavior].sum(axis=1) == df_raw['result']).all()
rule_match = (((df_raw['result'] >= 7).map({True:'YES',False:'NO'})) == df_raw['Class/ASD']).all()
print('A1~A10 합 = result:', sum_match)
print('result >= 7 ↔ Class/ASD YES:', rule_match)

A1~A10 합 = result: True
result >= 7 ↔ Class/ASD YES: True


### 결론
- `result`는 A1~A10 합계이며 Target 생성 규칙을 직접 포함하므로 독립 요인/모델 입력에서 제외합니다.
- `age_desc`는 모든 행에서 동일한 `4-11 years` 설명값이라 제외합니다.
- 실제 나이 `age`는 별도 컬럼이므로 분석 후보에 포함합니다.
- A1~A10 행동 문항 10개와 개인·배경 요인 8개, 총 18개 요인의 연관성을 다음 단계에서 확인합니다.
